# Segmentation Analysis

Checks whether the model's discrimination and calibration hold up *within*
key subgroups, not just in aggregate -- a single overall AUC/KS number (as
computed in `04_Model-Development`) can hide a segment where the model is
quietly worse, or where predicted risk is systematically mis-calibrated.

Runs on the same held-out validation set `04_Model-Development` already saved
(`test_predictions.parquet`), joined back to the demographic/segment
columns from `model_dataset_final.parquet` by `SK_ID_CURR` -- no re-scoring,
no model loading.

**Pipeline overview**
1. Configuration
2. Load test-set predictions + join segment attributes
3. Per-segment metrics (bad rate, mean PD, AUC, KS) with a minimum-sample-size guard
4. Run across every segment attribute
5. Visualize bad rate and AUC by segment
6. Save outputs

**Prerequisite:** `04_Model-Development` must have been run first -- it saves
`test_predictions.parquet`, which this notebook reads (same file
`06_Scoring`/`Credit_Scoring.ipynb` reads).

**Outputs**
- `data/Model-Training/ModelOutput/<model_folder>/SegmentationAnalysis/segment_summary.xlsx`
- `data/Model-Training/ModelOutput/<model_folder>/SegmentationAnalysis/bad_rate_by_segment.png`
- `data/Model-Training/ModelOutput/<model_folder>/SegmentationAnalysis/auc_by_segment.png`


## 1. Imports & Configuration

Same self-configuration pattern as `Credit_Scoring.ipynb` -- `MODEL_FOLDER`
is read from `HOMECREDIT_SCORING_MODEL_FOLDER` so the same env var already
used to point that notebook at a given model's outputs also points this
one, without a second variable to keep in sync.

`MIN_SEGMENT_SIZE` guards AUC/KS: a category with only a handful of
validation-set rows (some `ORGANIZATION_TYPE` values have well under 100)
gives a wildly noisy AUC that looks like a real problem but is really just
sampling noise -- those segments are flagged `reliable=False` instead of
computed.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from scipy import stats

# -- Paths -------------------------------------------------------------------
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")

# Which model's validation predictions to analyze -- same env var
# Credit_Scoring.ipynb uses, so one variable points both notebooks at
# whatever 04_Model-Development just produced.
MODEL_FOLDER = os.environ.get("HOMECREDIT_SCORING_MODEL_FOLDER", "LightGBM_v4")
MODEL_OUTPUT_DIR = os.path.join(DATA_ROOT, "Model-Training", "ModelOutput", MODEL_FOLDER)
PATH_TEST_PREDICTIONS = os.path.join(MODEL_OUTPUT_DIR, "test_predictions.parquet")
PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "Model-Training", "ModelDataset", "model_dataset_final.parquet")

SEGMENT_OUTPUT_DIR = os.path.join(MODEL_OUTPUT_DIR, "SegmentationAnalysis")
os.makedirs(SEGMENT_OUTPUT_DIR, exist_ok=True)

# Below this many rows in a segment, AUC/KS are too noisy to trust --
# flagged instead of computed. 30 is a conventional floor for a stable
# rate/AUC estimate, not a statistically derived cutoff.
MIN_SEGMENT_SIZE = 30

print(f"Model folder      : {MODEL_FOLDER}")
print(f"Test predictions  : {PATH_TEST_PREDICTIONS}")
print(f"Model dataset     : {PATH_MODEL_DATASET}")
print(f"Outputs to        : {SEGMENT_OUTPUT_DIR}")


## 2. Load Test-Set Predictions + Join Segment Attributes

`test_predictions.parquet` has `SK_ID_CURR`, `TARGET`, `predicted_proba` for
the held-out validation set -- no demographic columns. Those live in
`model_dataset_final.parquet` (the full pre-feature-selection dataset), so
this joins them back in by `SK_ID_CURR` for just the validation-set rows.

`HAS_BUREAU_HISTORY` mirrors the real-time API's thin-file case (see
`src/Inference/explainability.py` / `README.md`): whether
`BUREAUCREDIT_DAYS_TO_LAST_CREDIT` (one of the model's 24 features) is
populated for this applicant at all. A brand-new applicant with no Home
Credit history gets `NaN` there -- this segment checks whether the model's
discrimination actually holds up for that population, since roughly half
its total feature importance is unavailable to them (see README).

In [ ]:
predictions_df = pd.read_parquet(PATH_TEST_PREDICTIONS)
print(f"Loaded {len(predictions_df):,} validation predictions")

segment_cols = [
    "SK_ID_CURR", "CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE", "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
    "AGE", "BUREAUCREDIT_DAYS_TO_LAST_CREDIT",
]
segment_attrs = pd.read_parquet(PATH_MODEL_DATASET, columns=segment_cols)

segment_attrs["AGE_BAND"] = pd.cut(
    segment_attrs["AGE"], bins=[0, 25, 35, 45, 55, 65, 120],
    labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"],
)
segment_attrs["HAS_BUREAU_HISTORY"] = np.where(
    segment_attrs["BUREAUCREDIT_DAYS_TO_LAST_CREDIT"].notna(), "Yes", "No"
)

df = predictions_df.merge(segment_attrs, on="SK_ID_CURR", how="left")
print(f"Joined shape: {df.shape} (should still be {len(predictions_df):,} rows)")
assert len(df) == len(predictions_df), "join changed row count -- SK_ID_CURR mismatch between the two files"
df.head(3)


## 3. Per-Segment Metrics

For each category within a segment attribute: applicant count, share of the
validation set, actual bad rate, mean predicted PD (calibration -- these two
should track each other), AUC, and KS -- the last two only when the segment
has at least `MIN_SEGMENT_SIZE` rows and both classes present, otherwise
`reliable=False` and they're left as `NaN` rather than a noisy number.

An `ALL` row (the full validation set) is appended to every table as the
baseline every segment is compared against.

In [ ]:
def segment_metrics(frame: pd.DataFrame, segment_col: str, min_n: int = MIN_SEGMENT_SIZE) -> pd.DataFrame:
    rows = []
    for name, group in frame.groupby(segment_col, observed=True):
        n = len(group)
        reliable = n >= min_n and group["TARGET"].nunique() == 2
        auc = roc_auc_score(group["TARGET"], group["predicted_proba"]) if reliable else np.nan
        if reliable:
            good_pd = group.loc[group["TARGET"] == 0, "predicted_proba"]
            bad_pd = group.loc[group["TARGET"] == 1, "predicted_proba"]
            ks = stats.ks_2samp(bad_pd, good_pd).statistic
        else:
            ks = np.nan
        rows.append({
            "segment": str(name),
            "count": n,
            "volume_pct": n / len(frame),
            "bad_rate": group["TARGET"].mean(),
            "mean_pd": group["predicted_proba"].mean(),
            "auc": auc,
            "ks": ks,
            "reliable": reliable,
        })

    result = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
    overall_auc = roc_auc_score(frame["TARGET"], frame["predicted_proba"])
    overall = {
        "segment": "ALL", "count": len(frame), "volume_pct": 1.0,
        "bad_rate": frame["TARGET"].mean(), "mean_pd": frame["predicted_proba"].mean(),
        "auc": overall_auc,
        "ks": stats.ks_2samp(
            frame.loc[frame["TARGET"] == 1, "predicted_proba"],
            frame.loc[frame["TARGET"] == 0, "predicted_proba"],
        ).statistic,
        "reliable": True,
    }
    result = pd.concat([pd.DataFrame([overall]), result], ignore_index=True)

    # Heuristic flag, not a statistical test: a segment worth a second look
    # is one whose bad rate is well off the overall rate, or whose AUC is
    # meaningfully worse -- both computed only where reliable.
    overall_bad_rate = overall["bad_rate"]
    result["flag_bad_rate"] = result["reliable"] & (
        (result["bad_rate"] > overall_bad_rate * 1.5) | (result["bad_rate"] < overall_bad_rate * 0.5)
    ) & (result["segment"] != "ALL")
    result["flag_auc_drop"] = result["reliable"] & (result["auc"] < overall_auc - 0.05) & (result["segment"] != "ALL")
    return result


## 4. Run Across Every Segment Attribute

Demographics from the application form (`CODE_GENDER`, education, family
status, housing, income type, occupation, organization, age band) plus the
one behavioral segment that matters most for this model: whether the
applicant has any Home Credit bureau history at all.

In [ ]:
SEGMENT_COLUMNS = [
    "CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE",
    "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE", "AGE_BAND", "HAS_BUREAU_HISTORY",
]

segment_results = {col: segment_metrics(df, col) for col in SEGMENT_COLUMNS}

flagged = []
for col, result in segment_results.items():
    print(f"\n=== {col} ===")
    display_cols = ["segment", "count", "volume_pct", "bad_rate", "mean_pd", "auc", "ks", "reliable"]
    print(result[display_cols].to_string(index=False, float_format="{:.4f}".format))
    for _, row in result[(result["flag_bad_rate"]) | (result["flag_auc_drop"])].iterrows():
        flagged.append({"attribute": col, **row.to_dict()})

flagged_df = pd.DataFrame(flagged)
print(f"\n{len(flagged_df)} segment(s) flagged for a bad rate or AUC deviation from the overall validation set.")
if len(flagged_df):
    print(flagged_df[["attribute", "segment", "count", "bad_rate", "auc", "flag_bad_rate", "flag_auc_drop"]].to_string(index=False))


## 5. Visualize Bad Rate and AUC by Segment

One chart per segment attribute, sorted by count, with the overall
validation-set rate/AUC drawn as a reference line -- makes it easy to spot
a category sitting well off that line at a glance, rather than reading
every row of the printed tables above.

In [ ]:
def plot_metric_by_segment(result: pd.DataFrame, metric: str, title: str, ax, pct: bool = False):
    plot_df = result[result["segment"] != "ALL"].sort_values("count", ascending=True)
    if metric == "auc":
        plot_df = plot_df[plot_df["reliable"]]
    if plot_df.empty:
        ax.set_visible(False)
        return
    overall_value = result.loc[result["segment"] == "ALL", metric].iloc[0]
    colors = ["#C44E52" if f else "#4C72B0" for f in plot_df.get("flag_bad_rate" if metric == "bad_rate" else "flag_auc_drop", False)]
    ax.barh(plot_df["segment"], plot_df[metric], color=colors)
    ax.axvline(overall_value, color="black", linestyle="--", linewidth=1, label=f"Overall: {overall_value:.3f}")
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="y", labelsize=8)
    if pct:
        from matplotlib.ticker import PercentFormatter
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(fontsize=8, loc="lower right")


fig, axes = plt.subplots(len(SEGMENT_COLUMNS), 1, figsize=(9, 3.2 * len(SEGMENT_COLUMNS)))
for ax, col in zip(axes, SEGMENT_COLUMNS):
    plot_metric_by_segment(segment_results[col], "bad_rate", f"Bad rate by {col} (red = flagged)", ax, pct=True)
fig.tight_layout()
bad_rate_path = os.path.join(SEGMENT_OUTPUT_DIR, "bad_rate_by_segment.png")
fig.savefig(bad_rate_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {bad_rate_path}")

fig2, axes2 = plt.subplots(len(SEGMENT_COLUMNS), 1, figsize=(9, 3.2 * len(SEGMENT_COLUMNS)))
for ax, col in zip(axes2, SEGMENT_COLUMNS):
    plot_metric_by_segment(segment_results[col], "auc", f"AUC by {col} (red = flagged, unreliable segments omitted)", ax)
fig2.tight_layout()
auc_path = os.path.join(SEGMENT_OUTPUT_DIR, "auc_by_segment.png")
fig2.savefig(auc_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {auc_path}")


## 6. Save Outputs

One sheet per segment attribute plus a `Flagged` summary sheet, so a
reviewer can open one workbook rather than re-running this notebook.

In [ ]:
summary_path = os.path.join(SEGMENT_OUTPUT_DIR, "segment_summary.xlsx")
with pd.ExcelWriter(summary_path) as writer:
    for col, result in segment_results.items():
        result.to_excel(writer, sheet_name=col[:31], index=False)
    (flagged_df if len(flagged_df) else pd.DataFrame(columns=["attribute", "segment"])).to_excel(
        writer, sheet_name="Flagged", index=False
    )

print(f"Saved -> {summary_path}")
print("\nDone. Review the 'Flagged' sheet (or the red bars above) before promoting this model to @prod.")
